# КТ-2: предобработка, чанкинг и метаданные

## Цель

Для корпуса из **500 описаний изображений городских камер** строится воспроизводимый retrieval pipeline. В качестве релевантности используется gold-набор КТ-1: запрос считается успешным, если среди первых `k` результатов найдено изображение из `relevant_image_ids`.

В контрольной точке последовательно исследуются:

1. baseline без очистки: `TokenChunker(512)` + TF-IDF;
2. четыре стадии предобработки с измерением эффекта каждой;
3. `TokenChunker`, `SentenceChunker`, `SemanticChunker` и `NeuralChunker` из Chonkie;
4. ablation порога и модели эмбеддингов для семантического чанкинга;
5. parent/child retrieval с BGE-M3;
6. структурные, контентные и качественные метаданные;
7. сравнение извлечённых fact-chunks с чанкированием сырого описания (TF-IDF и BGE-M3);
8. LLM-summary и HyDE-вопросы;
9. итоговая сравнительная таблица.

Корпус необычно короткий: один image-parent document в среднем занимает около 400 токенов. Поэтому часть конфигураций закономерно даёт похожие границы. Это не ошибка эксперимента, а важное ограничение применимости чанкинга для данного корпуса.

## 0. Установка и конфигурация

В локальном checkout устанавливается текущий проект в editable-режиме. В чистой среде Google Colab устанавливается ветка [`v0_ct2`](https://github.com/sisgrad/e2e_rag_gigaschool_llm_course/tree/v0_ct2). Все зависимости, включая Chonkie, Transformers и Sentence Transformers, задаются в `pyproject.toml`.

Тяжёлые операции кэшируются в `evaluation/v0/output/cache`: повторный запуск не пересчитывает границы чанков, BGE-M3 embeddings и HyDE-тексты, если входы и конфигурация не изменились.

In [1]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/sisgrad/e2e_rag_gigaschool_llm_course"
REPOSITORY_REF = "v0_ct2"


def find_local_repository():
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / "pyproject.toml").is_file() and (directory / "src/gigaschool_rag").is_dir():
            return directory
    return None


local_repository = find_local_repository()
install_target = (
    str(local_repository)
    if local_repository is not None
    else f"git+{REPOSITORY_URL}.git@{REPOSITORY_REF}"
)
command = [sys.executable, "-m", "pip", "install", "--quiet"]
if local_repository is not None:
    command.extend(["--editable", install_target])
else:
    command.append(install_target)
subprocess.check_call(command)
print(f"Установлен проект: {install_target}")

Установлен проект: /app/TVA_VideoRetrieval/base_n_detailed_dataset_in_a_new_format/e2e_rag_gigaschool_llm_course


In [2]:
import html
import inspect
import os

import pandas as pd
import plotly.express as px
from IPython.display import HTML, Markdown, display

from gigaschool_rag.versions.v0 import resolve_dataset_root
from gigaschool_rag.versions.v0.control_point_2 import (
    add_metadata,
    chunk_documents,
    compare_chunkers,
    deduplicate_documents,
    evaluate_dense,
    evaluate_tfidf,
    fact_vs_chunking_experiment,
    generate_hypothetical_questions,
    llm_experiments,
    load_documents,
    load_fact_chunks,
    metadata_experiments,
    normalize_symbols,
    normalize_unicode_whitespace,
    parent_child_chunks,
    parent_child_experiment,
    prepend_metadata,
    preprocessing_experiment,
    run_control_point_2,
    semantic_ablation,
    strip_markup,
    validate_dataset_consistency,
)


def show_package_source(*functions):
    """Показать исходный код package-функций в компактном раскрываемом блоке."""
    blocks = []
    for function in functions:
        name = f"{function.__module__}.{function.__name__}"
        source = html.escape(inspect.getsource(function))
        blocks.append(
            f"<details><summary><code>{name}</code></summary>"
            f"<pre><code>{source}</code></pre></details>"
        )
    display(HTML("\n".join(blocks)))


show_package_source(resolve_dataset_root)
DATASET_ROOT = resolve_dataset_root()
if local_repository is not None:
    OUTPUT_DIR = local_repository / "evaluation/v0/output"
else:
    OUTPUT_DIR = Path("/content/gigaschool_rag_cp2_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_ROOT}")
print(f"Output/cache: {OUTPUT_DIR}")

Dataset: /app/TVA_VideoRetrieval/base_n_detailed_dataset_in_a_new_format/e2e_rag_gigaschool_llm_course/hf/v0
Output/cache: /app/TVA_VideoRetrieval/base_n_detailed_dataset_in_a_new_format/e2e_rag_gigaschool_llm_course/evaluation/v0/output


## 1. Дизайн эксперимента и метрики

Эксперименты проводятся последовательно, изменяя один основной фактор:

- **полный gold** используется для исходного baseline и демонстрации эффекта очистки;
- **validation split** используется для выбора стратегии чанкинга, порога и boundary-embedder;
- **test split** используется один раз для итоговых экспериментов с метаданными, LLM и parent/child.

Основные метрики:

- **Hit@k** — доля запросов, для которых среди первых `k` результатов найдено хотя бы одно релевантное изображение;
- **Recall@k** — средняя доля всех релевантных изображений, найденных в первых `k`;
- **Precision@k** — средняя доля релевантных изображений среди `k` результатов;
- **MAP** — средняя average precision с учётом позиций всех релевантных результатов.

Чанк оценивается на уровне родительского изображения: score изображения равен максимальному score его чанков. Негативные вопросы без релевантных изображений не включаются в retrieval-метрики, поскольку для них нужны отдельные abstention-метрики и порог score.

In [3]:
show_package_source(
    load_documents,
    strip_markup,
    normalize_unicode_whitespace,
    normalize_symbols,
    deduplicate_documents,
    preprocessing_experiment,
    evaluate_tfidf,
)

## 2. Выполнение воспроизводимого pipeline

Одна package-функция выполняет все стадии в фиксированной последовательности и сохраняет табличные артефакты. Первый запуск скачивает boundary-модели, NeuralChunker, BGE-M3 и локальную `google/flan-t5-small`; последующие запуски используют Hugging Face cache и собственный cache эксперимента.

NeuralChunker и HyDE выполняются и на CPU. BGE-M3 — модель примерно на 568 млн параметров — автоматически запускается только при доступном CUDA GPU. На CPU эксперимент не подменяется другой моделью: он явно пропускается, поскольку даже один embedding в данном контейнере занимает несколько минут. В Google Colab для полного прогона следует выбрать GPU runtime. Принудительный CPU-запуск возможен через `CP2_FORCE_BGE_CPU=1`, но практически не рекомендуется.

In [4]:
import time

import torch

show_package_source(validate_dataset_consistency, run_control_point_2)

RUN_BGE_M3 = torch.cuda.is_available() or os.getenv("CP2_FORCE_BGE_CPU") == "1"
if not RUN_BGE_M3:
    print("CUDA недоступна: BGE-M3 parent/child будет явно пропущен без подмены модели.")

started = time.perf_counter()
results = run_control_point_2(
    DATASET_ROOT,
    output_dir=OUTPUT_DIR,
    run_neural=True,
    run_dense=RUN_BGE_M3,
    run_hyde=True,
)
print(f"Pipeline завершён за {time.perf_counter() - started:.1f} секунд")
print(f"Выбрана semantic-конфигурация: {results['selected_semantic_configuration']}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Pipeline завершён за 19.3 секунд
Выбрана semantic-конфигурация: paraphrase-MiniLM-L3-v2@0.70


### 2.1. Проверка согласованности датасета и gold

Перед измерениями pipeline сверяет фактические размеры финальных JSONL с `manifest.json` и проверяет, что все image/fact ID из gold существуют. `provenance/agreement_summary.json` отражает более раннюю стадию producer/verifier agreement и содержит 7489 фактов, тогда как после нормализации в релизе осталось 7481. Поэтому источником истины для КТ-2 являются финальные `annotations/vlm_inferred/*.jsonl`; их 7481/5264/2217 совпадают с manifest.

In [5]:
consistency = results["dataset_consistency"]
display(pd.DataFrame({
    "final_jsonl": consistency["actual"],
    "manifest": consistency["manifest"],
    "agreement_provenance": consistency["agreement_provenance"],
}))
print("Manifest совпадает с JSONL:", consistency["manifest_matches_actual"])
print("Agreement summary совпадает с JSONL:", consistency["agreement_matches_actual"])
print("Отсутствующие gold image IDs:", len(consistency["missing_gold_image_ids"]))
print("Отсутствующие gold fact IDs:", len(consistency["missing_gold_fact_ids"]))

,final_jsonl,manifest,agreement_provenance
images,500,500,500
unique_facts,7481,7481,7489
trusted_facts,5264,5264,5271
untrusted_facts,2217,2217,2218


Manifest совпадает с JSONL: True
Agreement summary совпадает с JSONL: False
Отсутствующие gold image IDs: 0
Отсутствующие gold fact IDs: 0


## 3. Baseline и предобработка

Baseline строится **до любой очистки**: Chonkie `TokenChunker`, 512 токенов без overlap, TF-IDF unigram+bigram, агрегация `max` от чанка к изображению.

Далее строго последовательно применяются:

1. удаление HTML/Markdown, `script/style/nav`;
2. NFKC-нормализация Unicode, пробелов и переносов;
3. удаление управляющих символов и вербализация emoji вместо безусловного удаления;
4. exact + MinHash near-deduplication (`Jaccard ≥ 0.88`).

При дедупликации сохраняется список image aliases: два текстово одинаковых описания могут соответствовать разным изображениям и не должны исчезать из image-retrieval gold.

In [6]:
preprocessing = results["preprocessing"]
display(preprocessing.round(4))

metric_frame = preprocessing.melt(
    id_vars="stage",
    value_vars=["hit@5", "recall@5", "hit@10", "recall@10", "map"],
    var_name="metric",
    value_name="value",
)
display(px.line(
    metric_frame,
    x="stage",
    y="value",
    color="metric",
    markers=True,
    title="Retrieval-метрики после каждого шага предобработки",
))

display(px.bar(
    preprocessing,
    x="stage",
    y=["tokens", "documents"],
    barmode="group",
    title="Размер корпуса до и после каждого шага",
))
print("Отчёт дедупликации:", results["duplicate_report"])

,stage,documents,represented_images,tokens,mean_tokens,noise_score,hit@5,precision@5,recall@5,hit@10,precision@10,recall@10,map,input_documents,output_documents,exact_removed,near_removed,removed_percent,near_duplicate_threshold,image_aliases_preserved
0,raw,500,500,201321,402.642,0.0,0.95,0.545,0.0179,0.95,0.5525,0.0324,0.6273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,strip_markup,500,500,201321,402.642,0.0,0.95,0.545,0.0179,0.95,0.5525,0.0324,0.6273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,unicode_whitespace,500,500,201321,402.642,0.0,0.95,0.545,0.0179,0.95,0.5525,0.0324,0.6273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,emoji_symbols,500,500,201321,402.642,0.0,0.95,0.545,0.0179,0.95,0.5525,0.0324,0.6273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,deduplicate,500,500,201321,402.642,0.0,0.95,0.545,0.0179,0.95,0.5525,0.0324,0.6273,500.0,500.0,0.0,0.0,0.0,0.88,True


Отчёт дедупликации: {'input_documents': 500, 'output_documents': 500, 'exact_removed': 0, 'near_removed': 0, 'removed_percent': 0.0, 'near_duplicate_threshold': 0.88, 'image_aliases_preserved': True}


## 4. Сравнение четырёх Chonkie-чанкеров

- **TokenChunker(512)** — обязательный fixed-token baseline.
- **SentenceChunker(160)** — сохраняет границы предложений; меньший лимит выбран из-за коротких документов.
- **SemanticChunker(0.70)** — разрывает текст при семантическом сдвиге.
- **NeuralChunker** — предсказывает границы моделью `mirth/chonky_distilbert_base_uncased_1`.

Помимо retrieval измеряются среднее, медиана и стандартное отклонение длины, а также доли чанков короче 50 и длиннее 500 токенов. Screening выполняется только на validation split.

In [7]:
show_package_source(chunk_documents, compare_chunkers)
chunkers = results["chunkers"]
display(chunkers.round(4))

display(px.bar(
    chunkers,
    x="configuration",
    y=["hit@10", "recall@10", "map"],
    barmode="group",
    title="Validation: качество retrieval для четырёх принципов чанкинга",
))

display(px.scatter(
    chunkers,
    x="mean_chunk_tokens",
    y="recall@10",
    size="chunks",
    color="configuration",
    hover_data=["median_chunk_tokens", "std_chunk_tokens", "share_lt_50", "share_gt_500"],
    title="Компромисс между размером чанка и Recall@10",
))

,configuration,preprocessing,chunker,chunk_seconds,metadata,llm_processing,chunks,mean_chunk_tokens,median_chunk_tokens,std_chunk_tokens,share_lt_50,share_gt_500,hit@5,precision@5,recall@5,hit@10,precision@10,recall@10,map,index_seconds
0,TokenChunker-512,all documented stages,token,0.0110,none,none,580,347.1241,374.5,145.9254,0.0707,0.1569,0.9286,0.4929,0.0160,0.9286,0.4964,0.0272,0.6084,0.0828
1,SentenceChunker-160,all documented stages,sentence,0.0141,none,none,1594,126.2992,145.0,40.2729,0.1054,0.0000,0.7857,0.5357,0.0164,0.9643,0.5464,0.0304,0.6123,0.0887
2,SemanticChunker-0.70,all documented stages,semantic,0.0250,none,none,5230,38.4960,34.0,27.9042,0.7086,0.0000,0.9286,0.5500,0.0179,0.9643,0.5536,0.0312,0.5942,0.0972
3,NeuralChunker,all documented stages,neural,0.0186,none,none,3109,64.7543,52.0,46.2486,0.4754,0.0000,0.8214,0.4643,0.0152,0.9643,0.4893,0.0280,0.5906,0.0930


## 5. SemanticChunker: ablation

Сетка `0.55 / 0.70 / 0.85` покрывает низкую, среднюю и строгую cosine-similarity связность без подгонки под test. Для каждой точки сравниваются два boundary-embedder разного размера:

- `all-MiniLM-L6-v2`;
- `paraphrase-MiniLM-L3-v2`.

Порог и модель выбираются по `Recall@10`, затем `Hit@10` и MAP на validation. Таблица примеров ниже показывает один и тот же документ при разных настройках — это позволяет увидеть реальные границы, а не только агрегированные метрики.

In [8]:
show_package_source(semantic_ablation)
semantic = results["semantic_ablation"]
display(semantic.round(4))

display(px.line(
    semantic,
    x="threshold",
    y="recall@10",
    color="embedding_model",
    markers=True,
    hover_data=["mean_chunk_tokens", "hit@10", "map", "chunk_seconds"],
    title="Ablation: порог, boundary-embedder и Recall@10",
))

display(px.scatter(
    semantic,
    x="mean_chunk_tokens",
    y="recall@10",
    color="embedding_model",
    symbol="threshold",
    size="chunks",
    title="Как порог меняет размер чанка и retrieval",
))

display(Markdown(f"**Выбранная конфигурация:** `{results['selected_semantic_configuration']}`"))
display(results["semantic_examples"].head(30))

,configuration,embedding_model,threshold,chunk_seconds,chunks,mean_chunk_tokens,median_chunk_tokens,std_chunk_tokens,share_lt_50,share_gt_500,hit@5,precision@5,recall@5,hit@10,precision@10,recall@10,map,index_seconds
0,all-MiniLM-L6-v2@0.55,sentence-transformers/all-MiniLM-L6-v2,0.55,0.1000,4253,47.3393,41.0,34.3988,0.5991,0.0,0.8929,0.5357,0.0172,0.9286,0.5179,0.0295,0.5912,0.0986
1,all-MiniLM-L6-v2@0.70,sentence-transformers/all-MiniLM-L6-v2,0.70,0.0265,5230,38.4960,34.0,27.9042,0.7086,0.0,0.9286,0.5500,0.0179,0.9643,0.5536,0.0312,0.5942,0.0999
2,all-MiniLM-L6-v2@0.85,sentence-transformers/all-MiniLM-L6-v2,0.85,0.0285,6167,32.6470,25.0,23.1731,0.7946,0.0,0.9286,0.5143,0.0171,0.9643,0.5321,0.0301,0.5972,0.0984
3,paraphrase-MiniLM-L3-v2@0.55,sentence-transformers/paraphrase-MiniLM-L3-v2,0.55,0.0224,4148,48.5371,43.0,34.2756,0.5779,0.0,0.9286,0.6429,0.0203,1.0000,0.6179,0.0341,0.6055,0.0932
4,paraphrase-MiniLM-L3-v2@0.70,sentence-transformers/paraphrase-MiniLM-L3-v2,0.70,0.0255,5094,39.5230,36.0,28.0591,0.6908,0.0,0.9286,0.6357,0.0205,1.0000,0.6464,0.0367,0.6011,0.0975
5,paraphrase-MiniLM-L3-v2@0.85,sentence-transformers/paraphrase-MiniLM-L3-v2,0.85,0.0277,5996,33.5774,27.0,23.6557,0.7849,0.0,0.8929,0.6143,0.0160,1.0000,0.6036,0.0343,0.6032,0.1034


**Выбранная конфигурация:** `paraphrase-MiniLM-L3-v2@0.70`

,configuration,document_id,chunk_position,text
0,all-MiniLM-L6-v2@0.55,100__27-5Donskova3,0,A residential area contains a multi-storey res...
1,all-MiniLM-L6-v2@0.55,100__27-5Donskova3,1,Direct sunlight and shadows indicate sunny day...
2,all-MiniLM-L6-v2@0.55,100__27-5Donskova3,2,"Evidence: A multi-level residential building, ..."
3,all-MiniLM-L6-v2@0.55,100__27-5Donskova3,3,The scene is shown during the day. Evidence: T...
4,all-MiniLM-L6-v2@0.55,100__27-5Donskova3,4,Evidence: Direct sunlight produces distinct ca...
5,all-MiniLM-L6-v2@0.55,100__27-5Donskova3,5,A road turn is visible.
6,all-MiniLM-L6-v2@0.55,100__27-5Donskova3,6,Evidence: The roadway curves around a landscap...
7,all-MiniLM-L6-v2@0.55,100__27-5Donskova3,7,A general car parking area is visible. Evidenc...
8,all-MiniLM-L6-v2@0.55,100__27-5Donskova3,8,Evidence: A multi-level building with repeated...
9,all-MiniLM-L6-v2@0.55,102__DetskySadKlenovykvartal3,0,A low-elevation camera overlooks a residential...


## 6. Метаданные и два сценария применения

Для каждого чанка извлекаются:

- структурные: `published_at`, `source`, длина документа, позиция и число чанков;
- контентные: top-5 TF-IDF keywords, язык `en`, тип `camera-image description`;
- качественные: доля неалфавитных токенов и `trusted_ratio` verifier.

Проверяются два сценария на test split:

1. **quality filtering** — оставляются документы с `trusted_ratio` не ниже медианы; важна Precision до/после и возможная потеря Recall;
2. **metadata prepend** — source, content type, language и keywords добавляются перед индексируемым текстом.

Дата здесь отражает запуск producer, а не время съёмки; фильтровать по «свежести изображения» было бы методологически неверно. Источник у корпуса один, поэтому source-only filter также неинформативен.

In [9]:
show_package_source(add_metadata, prepend_metadata, metadata_experiments)
metadata = results["metadata"]
display(metadata.round(4))

display(px.bar(
    metadata,
    x="configuration",
    y=["precision@10", "recall@10", "hit@10"],
    barmode="group",
    title="Test: эффект фильтрации и metadata prepend",
))

example_chunk = results["metadata_chunks"]["metadata-prepend"][0]
display(Markdown("**Пример метаданных чанка**"))
display(pd.Series({
    key: example_chunk.get(key)
    for key in [
        "document_id", "published_at", "source", "document_length",
        "position", "chunk_count", "keywords", "language",
        "content_type", "noise_score", "trusted_ratio",
    ]
}).to_frame("value"))

,configuration,metadata,llm_processing,chunks,mean_chunk_tokens,median_chunk_tokens,std_chunk_tokens,share_lt_50,share_gt_500,hit@5,precision@5,recall@5,hit@10,precision@10,recall@10,map,index_seconds
0,metadata-control,stored only,none,5094,39.5230,36.0,28.0591,0.6908,0.0,1.0,0.7333,0.0222,1.0,0.7250,0.0448,0.6696,0.0993
1,metadata-quality-filter,trusted_ratio >= 0.722,none,2574,38.7428,36.0,27.1731,0.7055,0.0,1.0,0.7833,0.0228,1.0,0.7833,0.0457,0.4082,0.0491
2,metadata-prepend,prepend,none,5094,73.0220,70.0,28.0897,0.2636,0.0,1.0,0.6833,0.0213,1.0,0.6917,0.0436,0.6753,0.1393


**Пример метаданных чанка**

,value
document_id,100__27-5Donskova3
published_at,2026-09-27T21:43:25.805472+00:00
source,VLM producer+verifier
document_length,375
position,0
chunk_count,11
keywords,"[visible, building, evidence, associated, resi..."
language,en
content_type,camera-image description
noise_score,0.0


## 7. ParentDocumentRetriever с BGE-M3

Мелкий child индексируется моделью `BAAI/bge-m3`, а найденный результат возвращает родительское изображение. Сравниваются три пары `(parent, child)` в токенах: `(256, 64)`, `(384, 96)`, `(512, 128)`.

Размеры меньше типовых выбраны осознанно: документы этого корпуса в среднем короче 512 токенов. Пары покрывают агрессивное, среднее и консервативное разбиение. Отдельный график по типам вопросов показывает, выигрывает ли parent/child на factual, composite или abstract запросах.

In [10]:
show_package_source(parent_child_chunks, evaluate_dense, parent_child_experiment)
parent_child = results["parent_child"]
parent_by_type = results["parent_by_question_type"]
display(parent_child.round(4))

if not parent_child.empty:
    display(px.bar(
        parent_child,
        x="configuration",
        y=["hit@10", "recall@10", "map"],
        barmode="group",
        title="Test: простой чанкер и три parent/child конфигурации с BGE-M3",
    ))
    display(px.bar(
        parent_by_type[parent_by_type["k"] == 10],
        x="question_type",
        y="macro_recall",
        color="configuration",
        barmode="group",
        title="Recall@10 по типу gold-вопроса",
    ))
else:
    print("Эксперимент явно пропущен: CUDA недоступна. В Colab выберите GPU runtime.")

,configuration,chunker,embedding_model,parent_size,child_size,metadata,llm_processing,chunks,mean_chunk_tokens,median_chunk_tokens,...,share_lt_50,share_gt_500,hit@5,precision@5,recall@5,hit@10,precision@10,recall@10,map,index_seconds
0,dense-simple-best-semantic,simple-semantic,BAAI/bge-m3,NaN,NaN,stored only,none,5094,39.5230,36.0,...,0.6908,0.0,1.0,0.8000,0.0247,1.0,0.7917,0.0462,0.6625,11.3788
1,parent-256/child-64,parent-child,BAAI/bge-m3,256.0,64.0,stored only,none,3550,56.8493,64.0,...,0.2130,0.0,1.0,0.7500,0.0221,1.0,0.7583,0.0458,0.6986,0.0585
2,parent-384/child-96,parent-child,BAAI/bge-m3,384.0,96.0,stored only,none,2419,83.3580,96.0,...,0.1075,0.0,1.0,0.8000,0.0252,1.0,0.7417,0.0448,0.7080,0.0498
3,parent-512/child-128,parent-child,BAAI/bge-m3,512.0,128.0,stored only,none,1851,108.8855,128.0,...,0.1032,0.0,1.0,0.7833,0.0236,1.0,0.7833,0.0458,0.7114,0.0714


## 7.1. Извлечённые факты vs чанкирование сырого описания

В КТ-1 индекс строился из **детерминированных fact-chunks** (`sentence + Evidence`), по одному факту на чанк, с фильтром trusted. В разделах 4–7 этого блокнота чанкеры работали над **сырым image-parent текстом** (summary + все producer facts + evidence) без trust-фильтра.

Здесь оба представления сравниваются на **одном и том же test gold** и двух ретриверах:

- **TF-IDF** — как в КТ-1;
- **BGE-M3** — dense cosine, та же модель, что и в parent/child.

Корпуса:

| Корпус | Что индексируется |
|--------|-------------------|
| `fact-chunks-trusted` | offline one-fact-per-chunk, только trusted |
| `fact-chunks-all` | offline one-fact-per-chunk, все факты |
| `raw-TokenChunker-512` | сырое описание, fixed token chunks |
| `raw-SentenceChunker-160` | сырое описание, sentence chunks |
| `raw-best-semantic-*` | сырое описание, лучший SemanticChunker с validation |

Так видно, даёт ли структурированное извлечение фактов выигрыш относительно эвристического чанкирования полного текста, и сохраняется ли этот выигрыш при переходе с TF-IDF на dense embeddings.

In [11]:
show_package_source(load_fact_chunks, fact_vs_chunking_experiment)
fact_vs = results["fact_vs_chunking"]
fact_by_type = results["fact_vs_chunking_by_type"]
display(Markdown("**Размеры корпусов**"))
display(pd.Series(results["fact_details"]["corpora_sizes"]).to_frame("chunks"))
display(fact_vs.round(4))

if not fact_vs.empty:
    display(px.bar(
        fact_vs,
        x="corpus",
        y="recall@10",
        color="retriever",
        barmode="group",
        hover_data=["hit@10", "map", "chunks", "mean_chunk_tokens"],
        title="Test: Recall@10 — fact chunks vs raw-description chunking",
    ))
    display(px.bar(
        fact_vs,
        x="corpus",
        y="map",
        color="retriever",
        barmode="group",
        title="Test: MAP — fact chunks vs raw-description chunking",
    ))
    if not fact_by_type.empty:
        display(px.bar(
            fact_by_type[fact_by_type["k"] == 10],
            x="question_type",
            y="macro_recall",
            color="configuration",
            barmode="group",
            title="Recall@10 по типу вопроса: факты vs сырое чанкирование",
        ))
else:
    print("Сравнение не выполнено.")

**Размеры корпусов**

,chunks
fact-chunks-trusted,5264
fact-chunks-all,7481
raw-TokenChunker-512,580
raw-SentenceChunker-160,1594
raw-best-semantic-paraphrase-MiniLM-L3-v2@0.70,5094


,configuration,corpus,retriever,chunker,embedding_model,metadata,llm_processing,chunks,mean_chunk_tokens,median_chunk_tokens,...,share_lt_50,share_gt_500,hit@5,precision@5,recall@5,hit@10,precision@10,recall@10,map,index_seconds
0,fact-chunks-trusted/tfidf,fact-chunks-trusted,tfidf,fact-extracted,tfidf,none,none,5264,20.8589,21.0,...,1.0000,0.0000,1.0000,0.9000,0.0265,1.0,0.8750,0.0518,0.7841,0.0572
1,fact-chunks-trusted/bge-m3,fact-chunks-trusted,bge-m3,fact-extracted,BAAI/bge-m3,none,none,5264,20.8589,21.0,...,1.0000,0.0000,1.0000,0.8167,0.0249,1.0,0.7583,0.0439,0.7395,0.0239
2,fact-chunks-all/tfidf,fact-chunks-all,tfidf,fact-extracted,tfidf,none,none,7481,21.9742,21.0,...,1.0000,0.0000,1.0000,0.7833,0.0235,1.0,0.7083,0.0410,0.6917,0.1261
3,fact-chunks-all/bge-m3,fact-chunks-all,bge-m3,fact-extracted,BAAI/bge-m3,none,none,7481,21.9742,21.0,...,1.0000,0.0000,1.0000,0.7667,0.0230,1.0,0.7083,0.0407,0.6988,0.0259
4,raw-TokenChunker-512/tfidf,raw-TokenChunker-512,tfidf,raw-description,tfidf,none,none,580,347.1241,374.5,...,0.0707,0.1569,1.0000,0.6667,0.0225,1.0,0.6833,0.0444,0.6716,0.1295
5,raw-TokenChunker-512/bge-m3,raw-TokenChunker-512,bge-m3,raw-description,BAAI/bge-m3,none,none,580,347.1241,374.5,...,0.0707,0.1569,1.0000,0.8333,0.0240,1.0,0.8333,0.0483,0.7250,0.0199
6,raw-SentenceChunker-160/tfidf,raw-SentenceChunker-160,tfidf,raw-description,tfidf,none,none,1594,126.2992,145.0,...,0.1054,0.0000,0.9167,0.7000,0.0222,1.0,0.6167,0.0398,0.6611,0.1400
7,raw-SentenceChunker-160/bge-m3,raw-SentenceChunker-160,bge-m3,raw-description,BAAI/bge-m3,none,none,1594,126.2992,145.0,...,0.1054,0.0000,1.0000,0.8000,0.0220,1.0,0.8083,0.0468,0.7381,0.0217
8,raw-best-semantic-paraphrase-MiniLM-L3-v2@0.70...,raw-best-semantic-paraphrase-MiniLM-L3-v2@0.70,tfidf,raw-description,tfidf,none,none,5094,39.5230,36.0,...,0.6908,0.0000,1.0000,0.7333,0.0222,1.0,0.7250,0.0448,0.6696,0.1436
9,raw-best-semantic-paraphrase-MiniLM-L3-v2@0.70...,raw-best-semantic-paraphrase-MiniLM-L3-v2@0.70,bge-m3,raw-description,BAAI/bge-m3,none,none,5094,39.5230,36.0,...,0.6908,0.0000,1.0000,0.8000,0.0247,1.0,0.7917,0.0462,0.6625,0.0245


## 8. LLM-обработка чанков

### Summary replacement

В producer output уже есть сгенерированное GPT-резюме каждого изображения. Поэтому повторный вызов платной модели не нужен: rich document (`summary + facts + evidence`) заменяется существующим `description`. Поля producer usage и duration дают **верхнюю оценку** стоимости: исходный вызов создавал одновременно summary и структурированные facts, поэтому приписывать всю стоимость только summary нельзя.

### HyDE-style questions

Локальная instruction-модель `google/flan-t5-small` генерирует до трёх вопросов к каждому документу. Индексируются вопросы вместо исходного текста. Фиксируются input/output tokens, wall time и cache hit. Это реальная LLM-генерация, но малая модель выбрана для воспроизводимости в Colab; результат нельзя автоматически переносить на более сильные API-модели.

Гипотеза: summary может потерять редкие фактические детали, а вопросы способны усилить query-like формулировки, но обе трансформации могут ухудшить полноту для составных запросов.

In [12]:
show_package_source(generate_hypothetical_questions, llm_experiments)
llm = results["llm"]
display(llm.round(4))
display(pd.Series(results["generation_stats"], dtype="object").to_frame("value"))

display(px.bar(
    llm,
    x="configuration",
    y=["hit@10", "recall@10", "map"],
    barmode="group",
    title="Test: исходный текст, LLM-summary и HyDE",
))

display(px.scatter(
    llm,
    x="llm_tokens",
    y="recall@10",
    size="llm_seconds",
    color="configuration",
    hover_data=["hit@10", "map"],
    title="Качество относительно LLM-затрат",
))

,configuration,llm_processing,llm_seconds,llm_tokens,chunks,mean_chunk_tokens,median_chunk_tokens,std_chunk_tokens,share_lt_50,share_gt_500,hit@5,precision@5,recall@5,hit@10,precision@10,recall@10,map,index_seconds
0,llm-control-rich-document,none,0.000,0,5094,39.523,36.0,28.0591,0.6908,0.0,1.0000,0.7333,0.0222,1.0,0.7250,0.0448,0.6696,0.0977
1,llm-producer-summary,existing GPT producer summary,13940.826,15442512,500,73.554,73.0,17.3815,0.0640,0.0,0.9167,0.5500,0.0172,1.0,0.5417,0.0325,0.6428,0.0201
2,llm-hyde-questions,google/flan-t5-small,0.000,285824,500,45.052,39.5,21.5128,0.7320,0.0,1.0000,0.7167,0.0226,1.0,0.6500,0.0373,0.5079,0.0133


,value
model,google/flan-t5-small
cache_hit,True
seconds,0.0
input_tokens,234668
output_tokens,51156


## 9. Итоговая сравнительная таблица

Каждая строка — отдельная конфигурация. Смешивать validation и test при выборе победителя нельзя, поэтому `experiment_family` следует читать вместе с описанным выше дизайном: chunker screening и semantic ablation относятся к validation, финальные сценарии — к test.

Время индексирования включает fit TF-IDF либо загрузку/encoding dense-модели. `chunk_seconds` и `llm_seconds` вынесены отдельно, чтобы стоимость подготовки не скрывалась внутри retrieval.

In [13]:
final = results["final"].copy()
preferred_columns = [
    "experiment_family", "configuration", "corpus", "retriever",
    "preprocessing", "chunker", "embedding_model", "metadata",
    "llm_processing", "chunks", "mean_chunk_tokens", "median_chunk_tokens",
    "std_chunk_tokens", "share_lt_50", "share_gt_500", "hit@5",
    "precision@5", "recall@5", "hit@10", "precision@10", "recall@10",
    "map", "index_seconds", "chunk_seconds", "llm_seconds", "llm_tokens",
]
display(final[[column for column in preferred_columns if column in final]].round(4))

display(px.scatter(
    final,
    x="index_seconds",
    y="recall@10",
    color="experiment_family",
    hover_name="configuration",
    size="chunks",
    title="Итог: Recall@10 относительно времени индексирования",
))

,experiment_family,configuration,corpus,retriever,preprocessing,chunker,embedding_model,metadata,llm_processing,chunks,...,precision@5,recall@5,hit@10,precision@10,recall@10,map,index_seconds,chunk_seconds,llm_seconds,llm_tokens
0,chunker-screening,TokenChunker-512,NaN,NaN,all documented stages,token,NaN,none,none,580,...,0.4929,0.0160,0.9286,0.4964,0.0272,0.6084,0.0828,0.0110,NaN,NaN
1,chunker-screening,SentenceChunker-160,NaN,NaN,all documented stages,sentence,NaN,none,none,1594,...,0.5357,0.0164,0.9643,0.5464,0.0304,0.6123,0.0887,0.0141,NaN,NaN
2,chunker-screening,SemanticChunker-0.70,NaN,NaN,all documented stages,semantic,NaN,none,none,5230,...,0.5500,0.0179,0.9643,0.5536,0.0312,0.5942,0.0972,0.0250,NaN,NaN
3,chunker-screening,NeuralChunker,NaN,NaN,all documented stages,neural,NaN,none,none,3109,...,0.4643,0.0152,0.9643,0.4893,0.0280,0.5906,0.0930,0.0186,NaN,NaN
4,semantic-ablation,all-MiniLM-L6-v2@0.55,NaN,NaN,NaN,NaN,sentence-transformers/all-MiniLM-L6-v2,NaN,NaN,4253,...,0.5357,0.0172,0.9286,0.5179,0.0295,0.5912,0.0986,0.1000,NaN,NaN
5,semantic-ablation,all-MiniLM-L6-v2@0.70,NaN,NaN,NaN,NaN,sentence-transformers/all-MiniLM-L6-v2,NaN,NaN,5230,...,0.5500,0.0179,0.9643,0.5536,0.0312,0.5942,0.0999,0.0265,NaN,NaN
6,semantic-ablation,all-MiniLM-L6-v2@0.85,NaN,NaN,NaN,NaN,sentence-transformers/all-MiniLM-L6-v2,NaN,NaN,6167,...,0.5143,0.0171,0.9643,0.5321,0.0301,0.5972,0.0984,0.0285,NaN,NaN
7,semantic-ablation,paraphrase-MiniLM-L3-v2@0.55,NaN,NaN,NaN,NaN,sentence-transformers/paraphrase-MiniLM-L3-v2,NaN,NaN,4148,...,0.6429,0.0203,1.0000,0.6179,0.0341,0.6055,0.0932,0.0224,NaN,NaN
8,semantic-ablation,paraphrase-MiniLM-L3-v2@0.70,NaN,NaN,NaN,NaN,sentence-transformers/paraphrase-MiniLM-L3-v2,NaN,NaN,5094,...,0.6357,0.0205,1.0000,0.6464,0.0367,0.6011,0.0975,0.0255,NaN,NaN
9,semantic-ablation,paraphrase-MiniLM-L3-v2@0.85,NaN,NaN,NaN,NaN,sentence-transformers/paraphrase-MiniLM-L3-v2,NaN,NaN,5996,...,0.6143,0.0160,1.0000,0.6036,0.0343,0.6032,0.1034,0.0277,NaN,NaN


## 10. Выводы

Следующая ячейка формирует выводы непосредственно из полученных чисел, чтобы текст не расходился с повторным запуском. При интерпретации важно различать отсутствие эффекта и неудачу реализации: если HTML, emoji или exact duplicates отсутствуют в исходном VLM-корпусе, соответствующая очистка корректно не меняет метрики.

**Главный системный результат КТ-2 — критически низкий Recall@k во всех конфигурациях.** Hit@10 часто близок к 1.0: среди первых 10 изображений почти всегда есть хотя бы одно релевантное. Но macro Recall@10 остаётся порядка нескольких процентов, потому что у gold-запросов сотни релевантных камер, а top-10 физически покрывает лишь малую долю. Поэтому относительные выигрыши чанкеров, метаданных и BGE-M3 нельзя читать как «задача решена»: они меняют ранжирование внутри всё ещё очень неполного покрытия.

Главный критерий практической рекомендации — не максимум одной метрики, а качество относительно сложности, времени и стоимости. Для короткого структурированного корпуса простой chunking может оказаться рациональнее тяжёлого semantic/parent-child pipeline, но ни один из вариантов не поднимает Recall до практически приемлемого уровня без увеличения `k`, фильтрации по labels или другого протокола оценки.

In [14]:
baseline = preprocessing.iloc[0]
after_cleaning = preprocessing.iloc[-1]
best_validation = chunkers.sort_values(["recall@10", "hit@10", "map"], ascending=False).iloc[0]
best_test = final[final["experiment_family"].isin([
    "metadata", "llm-processing", "parent-child", "fact-vs-raw-chunking"
])].sort_values(
    ["recall@10", "hit@10", "map"], ascending=False
).iloc[0]

from gigaschool_rag.versions.v0.control_point_2 import gold_queries

recall_pool = final["recall@10"].dropna()
max_recall = float(recall_pool.max()) if not recall_pool.empty else float("nan")
median_recall = float(recall_pool.median()) if not recall_pool.empty else float("nan")
mean_gt = float(
    pd.Series(
        [len(q["relevant_image_ids"]) for q in gold_queries(split="test")]
    ).mean()
)

lines = [
    "### Автоматически рассчитанное резюме",
    (
        f"- **Критически низкий Recall:** по всем конфигурациям финальной таблицы "
        f"Recall@10 в диапазоне [{recall_pool.min():.3f}; {max_recall:.3f}] "
        f"(медиана {median_recall:.3f}). Даже лучший вариант возвращает лишь около "
        f"{100 * max_recall:.1f}% релевантных изображений в top-10."
    ),
    (
        f"- Причина низкой полноты структурная: у test-запросов в среднем "
        f"~{mean_gt:.0f} релевантных изображений, поэтому top-10 покрывает не больше "
        f"~{100 * 10 / max(mean_gt, 1):.1f}% даже при идеальном ранжировании. "
        f"Высокий Hit@10 при таком Recall означает «хоть один релевантный результат есть», "
        f"а не «корпус хорошо покрыт»."
    ),
    f"- Baseline: Hit@10={baseline['hit@10']:.3f}, Recall@10={baseline['recall@10']:.3f}, MAP={baseline['map']:.3f}.",
    f"- После всех шагов очистки: ΔRecall@10={after_cleaning['recall@10'] - baseline['recall@10']:+.3f}; удалено документов: {int(baseline['documents'] - after_cleaning['documents'])}.",
    f"- Лучший чанкер на validation: **{best_validation['configuration']}**, Recall@10={best_validation['recall@10']:.3f}.",
    f"- Лучшая финальная конфигурация на test по Recall@10: **{best_test['configuration']}**, Recall@10={best_test['recall@10']:.3f} — относительно лучше других, но абсолютная полнота остаётся критически низкой.",
]

if not fact_vs.empty:
    for retriever in sorted(fact_vs["retriever"].dropna().unique()):
        subset = fact_vs[fact_vs["retriever"] == retriever]
        best_fact = subset[subset["corpus"].str.startswith("fact-")].sort_values(
            ["recall@10", "map"], ascending=False
        )
        best_raw = subset[subset["corpus"].str.startswith("raw-")].sort_values(
            ["recall@10", "map"], ascending=False
        )
        if best_fact.empty or best_raw.empty:
            continue
        fact_row = best_fact.iloc[0]
        raw_row = best_raw.iloc[0]
        lines.append(
            f"- {retriever}: лучшие факты **{fact_row['corpus']}** "
            f"(Recall@10={fact_row['recall@10']:.3f}, MAP={fact_row['map']:.3f}) vs "
            f"лучшее сырое чанкирование **{raw_row['corpus']}** "
            f"(Recall@10={raw_row['recall@10']:.3f}, MAP={raw_row['map']:.3f}); "
            f"ΔRecall@10={fact_row['recall@10'] - raw_row['recall@10']:+.3f}."
        )

if not parent_by_type.empty:
    simple = parent_by_type[(parent_by_type["configuration"] == "dense-simple-best-semantic") & (parent_by_type["k"] == 10)].set_index("question_type")
    candidates = parent_by_type[(parent_by_type["configuration"] != "dense-simple-best-semantic") & (parent_by_type["k"] == 10)]
    best_parent = candidates.sort_values("macro_recall", ascending=False).groupby("question_type").first()
    wins = []
    for question_type in sorted(set(simple.index) & set(best_parent.index)):
        delta = best_parent.loc[question_type, "macro_recall"] - simple.loc[question_type, "macro_recall"]
        wins.append(f"{question_type}: {delta:+.3f}")
    lines.append("- Parent/child против простого semantic chunker с тем же BGE-M3, ΔRecall@10: " + "; ".join(wins) + ".")

hyde_row = llm[llm["configuration"] == "llm-hyde-questions"]
if not hyde_row.empty:
    control_row = llm[llm["configuration"] == "llm-control-rich-document"].iloc[0]
    hyde_row = hyde_row.iloc[0]
    lines.append(
        f"- HyDE: ΔRecall@10={hyde_row['recall@10'] - control_row['recall@10']:+.3f}, "
        f"затраты={int(hyde_row['llm_tokens'])} токенов и {hyde_row['llm_seconds']:.1f} с."
    )

lines.append(
    "- Практический вывод: относительные улучшения чанкинга/метаданных/dense полезны для "
    "ранжирования, но **не снимают проблему критически низкой полноты** при оценке на "
    f"широких gold-запросах с ~{mean_gt:.0f} релевантными изображениями и бюджетом top-10."
)

display(Markdown("\n".join(lines)))

### Автоматически рассчитанное резюме
- **Критически низкий Recall:** по всем конфигурациям финальной таблицы Recall@10 в диапазоне [0.027; 0.052] (медиана 0.042). Даже лучший вариант возвращает лишь около 5.2% релевантных изображений в top-10.
- Причина низкой полноты структурная: у test-запросов в среднем ~217 релевантных изображений, поэтому top-10 покрывает не больше ~4.6% даже при идеальном ранжировании. Высокий Hit@10 при таком Recall означает «хоть один релевантный результат есть», а не «корпус хорошо покрыт».
- Baseline: Hit@10=0.950, Recall@10=0.032, MAP=0.627.
- После всех шагов очистки: ΔRecall@10=+0.000; удалено документов: 0.
- Лучший чанкер на validation: **SemanticChunker-0.70**, Recall@10=0.031.
- Лучшая финальная конфигурация на test по Recall@10: **fact-chunks-trusted/tfidf**, Recall@10=0.052 — относительно лучше других, но абсолютная полнота остаётся критически низкой.
- bge-m3: лучшие факты **fact-chunks-trusted** (Recall@10=0.044, MAP=0.740) vs лучшее сырое чанкирование **raw-TokenChunker-512** (Recall@10=0.048, MAP=0.725); ΔRecall@10=-0.004.
- tfidf: лучшие факты **fact-chunks-trusted** (Recall@10=0.052, MAP=0.784) vs лучшее сырое чанкирование **raw-best-semantic-paraphrase-MiniLM-L3-v2@0.70** (Recall@10=0.045, MAP=0.670); ΔRecall@10=+0.007.
- Parent/child против простого semantic chunker с тем же BGE-M3, ΔRecall@10: abstract: +0.011; composite: -0.005; factual: +0.003.
- HyDE: ΔRecall@10=-0.008, затраты=285824 токенов и 0.0 с.
- Практический вывод: относительные улучшения чанкинга/метаданных/dense полезны для ранжирования, но **не снимают проблему критически низкой полноты** при оценке на широких gold-запросах с ~217 релевантными изображениями и бюджетом top-10.